# Coupling simulator parameter sliders

Generate fixed participant histories, repeat each history across many independent simulator rounds, adjust the simulator parameters with sliders, and redraw the source arrows with the final-trial simulated choice distribution from `simulate_coupling`.

In [ ]:
import os
from pathlib import Path
import sys

MPLCONFIGDIR = Path("/tmp/coupling_sources_matplotlib")
MPLCONFIGDIR.mkdir(parents=True, exist_ok=True)
os.environ.setdefault("MPLCONFIGDIR", str(MPLCONFIGDIR))

import jax.numpy as jnp
import matplotlib.pyplot as plt
import numpy as np
from IPython.display import display
from ipywidgets import BoundedIntText, FloatSlider, HBox, VBox, interactive_output
from matplotlib.lines import Line2D

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "coupling_sources").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from coupling_sources.core.data import StimulusBatch, StimulusMetadata, StimulusSpec
from coupling_sources.core.simulator import simulate_coupling
from coupling_sources.core.stimuli import generate_stimulus

plt.rcParams["figure.figsize"] = (12, 6)
plt.rcParams["axes.facecolor"] = "white"

In [ ]:
N_GENERATED_PARTICIPANTS = 30
N_RESPONSE_REPEATS = 500
RESPONSE_SEED = 201


def repeat_single_round(batch, repeats):
    return StimulusBatch(
        visible_reports=jnp.repeat(batch.visible_reports, repeats, axis=1),
        hidden_source=jnp.repeat(batch.hidden_source, repeats, axis=1),
        trial_type=jnp.repeat(batch.trial_type, repeats, axis=1),
        valid_trials=jnp.repeat(batch.valid_trials, repeats, axis=1),
        metadata=StimulusMetadata(
            complete_reports=jnp.repeat(batch.metadata.complete_reports, repeats, axis=1),
            true_direction=jnp.repeat(batch.metadata.true_direction, repeats, axis=1),
            source_choices=jnp.repeat(batch.metadata.source_choices, repeats, axis=1),
            stimulus_covariance=batch.metadata.stimulus_covariance,
        ),
    )


STIMULUS_SPEC = StimulusSpec(
    seed=101,
    n_participants=N_GENERATED_PARTICIPANTS,
    n_sources=4,
    n_rounds=1,
    n_trials_full=20,
    n_trials_partial=20,
    sigma=0.7,
    lam=0.8,
    min_mean_angle_difference=jnp.pi / 4,
    coupled_sources=(0, 1),
)

base_stimuli = generate_stimulus(STIMULUS_SPEC)
stimuli = repeat_single_round(base_stimuli, repeats=N_RESPONSE_REPEATS)
N_PARTICIPANTS, N_ROUNDS, N_TRIALS, N_SOURCES = stimuli.visible_reports.shape
FINAL_TRIAL_INDEX = N_TRIALS - 1
SOURCE_COLORS = ["#2563eb", "#16a34a", "#f97316", "#7c3aed", "#0891b2", "#be123c"]
SOURCE_LABELS = [f"S{source}" for source in range(N_SOURCES)]

print(
    "stimuli:",
    f"participants={N_PARTICIPANTS}",
    f"generated_rounds={STIMULUS_SPEC.n_rounds}",
    f"repeated_rounds={N_ROUNDS}",
    f"trials={N_TRIALS}",
    f"sources={N_SOURCES}",
)

In [ ]:
def angle_xy(angle, radius=1.0):
    return radius * np.cos(angle), radius * np.sin(angle)


def circular_mean_angle(angles):
    angles = np.asarray(angles)
    finite = np.isfinite(angles)
    if not finite.any():
        return np.nan
    x = np.cos(angles[finite]).mean()
    y = np.sin(angles[finite]).mean()
    if np.isclose(x, 0.0) and np.isclose(y, 0.0):
        return np.nan
    return np.arctan2(y, x) % (2.0 * np.pi)


def draw_angle_arrow(
    ax,
    angle,
    *,
    color,
    radius=0.92,
    linewidth=2.6,
    linestyle="-",
    alpha=1.0,
    zorder=4,
):
    if not np.isfinite(angle):
        return
    x, y = angle_xy(angle, radius)
    ax.annotate(
        "",
        xy=(x, y),
        xytext=(0.0, 0.0),
        arrowprops={
            "arrowstyle": "-|>",
            "color": color,
            "lw": linewidth,
            "linestyle": linestyle,
            "mutation_scale": 16,
            "shrinkA": 0,
            "shrinkB": 0,
            "alpha": alpha,
        },
        zorder=zorder,
    )


def draw_circle_frame(ax):
    theta = np.linspace(0.0, 2.0 * np.pi, 361)
    ax.plot(np.cos(theta), np.sin(theta), color="#d1d5db", linewidth=1.4)
    ax.axhline(0.0, color="#e5e7eb", linewidth=0.8, zorder=0)
    ax.axvline(0.0, color="#e5e7eb", linewidth=0.8, zorder=0)
    ax.scatter([0.0], [0.0], s=20, color="#111827", zorder=5)
    ax.set_aspect("equal", adjustable="box")
    ax.set_xlim(-1.28, 1.28)
    ax.set_ylim(-1.28, 1.28)
    ax.axis("off")


def visible_label(source, is_hidden):
    if is_hidden:
        return f"{SOURCE_LABELS[source]} hidden"
    return SOURCE_LABELS[source]


def draw_simulation(alpha, gain_y, gain_z, kappa, c0, participant):
    participant = int(participant)
    history_round = 0
    trial_index = FINAL_TRIAL_INDEX
    parameters = {
        "alpha": float(alpha),
        "gain_y": float(gain_y),
        "gain_z": float(gain_z),
        "kappa": float(kappa),
        "c0": float(c0),
    }
    simulation = simulate_coupling(
        seed=RESPONSE_SEED,
        batch=stimuli,
        parameters=parameters,
    )

    visible_reports = np.asarray(stimuli.visible_reports[participant, history_round, trial_index])
    complete_reports = np.asarray(stimuli.metadata.complete_reports[participant, history_round, trial_index])
    true_direction = float(np.asarray(stimuli.metadata.true_direction[participant, history_round, trial_index]))
    hidden_source = int(np.asarray(stimuli.hidden_source[participant, history_round, trial_index]))
    is_probe = hidden_source >= 0

    diagnostics = simulation.diagnostics
    responses = np.asarray(simulation.responses[participant, :, trial_index])
    weights_name = "probe_weights" if is_probe else "truth_weights"
    weights = np.asarray(getattr(diagnostics, weights_name)[participant, history_round, trial_index])
    choice_mean = circular_mean_angle(responses)

    fig, (ax_circle, ax_weights) = plt.subplots(
        1,
        2,
        figsize=(12, 6),
        gridspec_kw={"width_ratios": [1.25, 0.9]},
    )
    draw_circle_frame(ax_circle)

    finite_responses = np.isfinite(responses)
    if finite_responses.any():
        response_samples = responses[finite_responses]
        jitter = (np.arange(response_samples.size) * 0.61803398875) % 1.0
        sample_radius = 1.01 + 0.20 * jitter
        xs = sample_radius * np.cos(response_samples)
        ys = sample_radius * np.sin(response_samples)
        ax_circle.scatter(
            xs,
            ys,
            s=26,
            color="#ef4444",
            alpha=0.28,
            linewidths=0,
            zorder=2,
        )

    if not is_probe:
        draw_angle_arrow(ax_circle, true_direction, color="#374151", radius=1.0, linewidth=2.0, zorder=3)

    for source, angle in enumerate(complete_reports):
        source_color = SOURCE_COLORS[source % len(SOURCE_COLORS)]
        source_hidden = source == hidden_source
        source_visible = np.isfinite(visible_reports[source])
        draw_angle_arrow(
            ax_circle,
            angle,
            color=source_color,
            radius=0.78,
            linewidth=3.0 if source_visible else 2.0,
            linestyle="--" if source_hidden else "-",
            alpha=1.0 if source_visible else 0.42,
            zorder=6 if source_visible else 2,
        )
        label_radius = 0.88 if source_visible else 0.68
        label_x, label_y = angle_xy(angle, radius=label_radius)
        ax_circle.text(
            label_x,
            label_y,
            visible_label(source, source_hidden and not source_visible),
            color=source_color,
            fontsize=10,
            ha="center",
            va="center",
            weight="bold" if source_visible else "normal",
            alpha=1.0 if source_visible else 0.65,
        )

    draw_angle_arrow(ax_circle, choice_mean, color="#dc2626", radius=1.0, linewidth=2.4, linestyle="-", alpha=0.8, zorder=9)

    trial_kind = "hidden-source probe" if is_probe else "truth estimate"
    choice_mean_label = "mean hidden-source prediction" if is_probe else "mean truth estimate"
    ax_circle.set_title(
        f"Participant {participant}, final trial {trial_index}: {trial_kind}",
        pad=12,
    )

    legend_items = []
    if not is_probe:
        legend_items.append(Line2D([0], [0], color="#374151", lw=2, label="true direction"))
    legend_items.extend([
        Line2D([0], [0], marker="o", color="none", markerfacecolor="#ef4444", markersize=7, alpha=0.45, label="simulated choices"),
        Line2D([0], [0], color="#dc2626", lw=2.4, label=choice_mean_label),
    ])
    if is_probe:
        legend_items.append(Line2D([0], [0], color="#9ca3af", lw=2, linestyle="--", label="hidden source"))
    ax_circle.legend(handles=legend_items, loc="lower center", bbox_to_anchor=(0.5, -0.08), ncol=2, frameon=False)

    x = np.arange(N_SOURCES)
    bar_colors = [SOURCE_COLORS[source % len(SOURCE_COLORS)] for source in range(N_SOURCES)]
    ax_weights.bar(x, weights, color=bar_colors, alpha=0.85)
    ax_weights.set_xticks(x, SOURCE_LABELS)
    ax_weights.set_ylim(0.0, 1.0)
    ax_weights.set_ylabel("weight")
    ax_weights.set_title(f"{weights_name.replace('_', ' ')} for this trial")
    ax_weights.grid(axis="y", alpha=0.25)
    if is_probe:
        ax_weights.axvline(hidden_source, color="#111827", linewidth=1.2, linestyle="--", alpha=0.55)
    for source, weight in enumerate(weights):
        ax_weights.text(source, min(float(weight) + 0.03, 0.96), f"{weight:.2f}", ha="center", va="bottom", fontsize=9)

    fig.suptitle(
        "alpha={alpha:.2f}, gain_y={gain_y:.2f}, gain_z={gain_z:.2f}, "
        "kappa={kappa:.1f}, c0={c0:.2f}, repeats={repeats}".format(
            alpha=alpha,
            gain_y=gain_y,
            gain_z=gain_z,
            kappa=kappa,
            c0=c0,
            repeats=N_RESPONSE_REPEATS,
        ),
        y=1.02,
    )
    fig.tight_layout()
    plt.show()


slider_style = {"description_width": "initial"}
slider_layout = {"width": "540px"}

alpha_slider = FloatSlider(value=0.20, min=0.0, max=1.0, step=0.01, description="alpha", continuous_update=False, readout_format=".2f", style=slider_style, layout=slider_layout)
gain_y_slider = FloatSlider(value=1.0, min=0.0, max=8.0, step=0.05, description="gain_y", continuous_update=False, readout_format=".2f", style=slider_style, layout=slider_layout)
gain_z_slider = FloatSlider(value=1.0, min=0.0, max=8.0, step=0.05, description="gain_z", continuous_update=False, readout_format=".2f", style=slider_style, layout=slider_layout)
kappa_slider = FloatSlider(value=8.0, min=0.1, max=64.0, step=0.1, description="kappa", continuous_update=False, readout_format=".1f", style=slider_style, layout=slider_layout)
c0_slider = FloatSlider(value=0.0, min=-1.0, max=1.0, step=0.01, description="c0", continuous_update=False, readout_format=".2f", style=slider_style, layout=slider_layout)

participant_counter = BoundedIntText(
    value=0,
    min=0,
    max=N_PARTICIPANTS - 1,
    step=1,
    description="participant",
    style=slider_style,
    layout={"width": "180px"},
)

controls = HBox([
    VBox([alpha_slider, gain_y_slider, gain_z_slider, kappa_slider, c0_slider]),
    VBox([participant_counter]),
], layout={"align_items": "flex-start"})
out = interactive_output(
    draw_simulation,
    {
        "alpha": alpha_slider,
        "gain_y": gain_y_slider,
        "gain_z": gain_z_slider,
        "kappa": kappa_slider,
        "c0": c0_slider,
        "participant": participant_counter,
    },
)
display(VBox([controls, out]))